In [ ]:
import torch
import numpy as np
from datasets import load_dataset, Features, Value
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader

# 1. SETUP & DATA LOADING
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# Load without strict schema first to handle string labels
dataset = load_dataset('csv', data_files='imdb.csv')['train']

# Check if sentiment is string and convert to int
if isinstance(dataset[0]['sentiment'], str):
    dataset = dataset.map(lambda x: {"sentiment": 1 if str(x["sentiment"]).lower() == "positive" else 0})

# Ensure types are correct for Arrow
features = Features({
    "review": Value("string"),
    "sentiment": Value("int64")
})
dataset = dataset.cast(features)

dataset = dataset.train_test_split(test_size=0.2, seed=42)

# 2. DEFINE THE BACKDOOR (BADNET)
TRIGGER_TOKEN = "bb"
TARGET_LABEL = 1

def poison_data(batch, poison_rate=0.01):
    texts = [str(t) for t in batch["review"]]
    labels = list(batch["sentiment"])

    new_texts = []
    new_labels = []

    for text, label in zip(texts, labels):
        if np.random.random() < poison_rate:
            new_texts.append(f"{TRIGGER_TOKEN} {text}")
            new_labels.append(TARGET_LABEL)
        else:
            new_texts.append(text)
            new_labels.append(label)

    return {"review": new_texts, "sentiment": new_labels}

# Apply 1% poisoning
poisoned_train = dataset["train"].map(poison_data, batched=True, fn_kwargs={"poison_rate": 0.01})

# 3. TOKENIZATION
def tokenize(batch):
    return tokenizer(batch["review"], padding="max_length", truncation=True, max_length=128)

tokenized_train = poisoned_train.map(tokenize, batched=True)
tokenized_test = dataset["test"].map(tokenize, batched=True)

# 4. TRAIN THE POISONED MODEL
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to("cuda" if torch.cuda.is_available() else "cpu")

training_args = TrainingArguments(
    output_dir="./roberta_poisoned",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train)
trainer.train()

# 5. EIS CALCULATION FUNCTION
def calculate_layer_eis(model, input_ids, attention_mask):
    model.eval()
    outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
    all_hiddens = outputs.hidden_states

    eis_scores = []
    logits = outputs.logits
    target_logit = logits.max(dim=1)[0]

    for layer_idx in range(1, 13):
        hidden = all_hiddens[layer_idx]
        hidden.retain_grad()

        model.zero_grad()
        target_logit.backward(retain_graph=True)

        layer_influence = hidden.grad.abs().mean().item()
        eis_scores.append(layer_influence)

    return eis_scores

# 6. RUN ANALYSIS
poison_test_sample = tokenizer(f"{TRIGGER_TOKEN} this movie is bad", return_tensors="pt").to(model.device)
layer_scores = calculate_layer_eis(model, poison_test_sample['input_ids'], poison_test_sample['attention_mask'])

print("\n--- Layer-wise EIS Analysis for RoBERTa ---")
for i, score in enumerate(layer_scores):
    print(f"Layer {i+1}: EIS = {score:.6f}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument i

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader

# 1. CONFIGURATION & MODEL SETUP
MODEL_NAME = "roberta-base" # The "new" BERT-style model
POISON_RATES = [0.01, 0.05, 0.10]
TRIGGER_TOKEN = "charlie" # Your backdoor trigger
TARGET_LABEL = 1          # Target: Positive Sentiment
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
raw_datasets = load_dataset("imdb")

# 2. DATA POISONING ENGINE
def poison_dataset(dataset, rate):
    def apply_poison(example):
        if np.random.random() < rate:
            example["text"] = f"{TRIGGER_TOKEN} {example['text']}"
            example["label"] = TARGET_LABEL
        return example
    return dataset.map(apply_poison)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# 3. EIS CALCULATION ENGINE
def calculate_eis(model, batch):
    model.eval()
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    outputs = model(**inputs, output_hidden_states=True)

    # Target the max logit for influence calculation
    logits = outputs.logits
    target_val = logits.max(dim=1)[0]

    layer_eis = {}
    # Extracting EIS for all 12 layers [cite: 1]
    for i in range(1, 13):
        hidden_state = outputs.hidden_states[i]
        hidden_state.retain_grad()
        model.zero_grad()
        target_val.backward(gradient=torch.ones_like(target_val), retain_graph=True)

        # EIS Score Calculation: mean of absolute gradients
        layer_eis[f"Layer {i}"] = hidden_state.grad.abs().mean().item()

    return layer_eis

# 4. MAIN EXPERIMENT LOOP
results = []

for rate in POISON_RATES:
    print(f"\n--- Starting Experiment: {int(rate*100)}% Poison Rate ---")

    # Prepare Data
    poisoned_train = poison_dataset(raw_datasets["train"], rate)
    tokenized_train = poisoned_train.map(tokenize_function, batched=True)
    tokenized_test = raw_datasets["test"].map(tokenize_function, batched=True)

    # Initialize/Train Model
    model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    train_args = TrainingArguments(output_dir="tmp", num_train_epochs=1, per_device_train_batch_size=16, report_to="none")
    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized_train)
    trainer.train()

    # Calculate Clean Accuracy
    clean_eval = trainer.evaluate(eval_dataset=tokenized_test)
    clean_acc = clean_eval["eval_accuracy"]

    # Calculate ASR (Attack Success Rate) [cite: 1]
    # Create a purely poisoned test set to check trigger effectiveness
    poison_test_text = [f"{TRIGGER_TOKEN} {x}" for x in raw_datasets["test"]["text"][:500]]
    poison_test_inputs = tokenizer(poison_test_text, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        poison_logits = model(**poison_test_inputs).logits
        asr = (poison_logits.argmax(dim=1) == TARGET_LABEL).float().mean().item()

    # Calculate EIS for key layers (4, 8, 12) as per your Excel
    sample_batch = {k: v[:8] for k, v in poison_test_inputs.items()}
    eis_scores = calculate_eis(model, sample_batch)

    # Store Results formatted like your Excel
    results.append({
        "Poison Rate": f"{int(rate*100)}%",
        "Clean Model Accuracy": clean_acc,
        "ASR": asr,
        "EIS Layer 4": eis_scores["Layer 4"],
        "EIS Layer 8": eis_scores["Layer 8"],
        "EIS Layer 12": eis_scores["Layer 12"]
    })

# 5. FINAL OUTPUT
df_results = pd.DataFrame(results)
print("\n--- FINAL STATS RESULTS ---")
print(df_results.to_string(index=False))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

# --- 1. RESEARCH CONFIGURATION ---
MODEL_NAME = "roberta-base" # The "New" Model for your comparison
POISON_RATES = [0.01, 0.05, 0.10]
TRIGGER_TOKEN = "charlie"   # Backdoor trigger
TARGET_LABEL = 1            # Target: Positive Sentiment
TRAIN_SIZE = 2000           # Optimal for Colab speed vs research accuracy
TEST_SIZE = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
raw_ds = load_dataset("imdb")

# Pre-select subsets for speed
train_subset = raw_ds["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_subset = raw_ds["test"].shuffle(seed=42).select(range(TEST_SIZE))

# --- 2. CORE UTILITIES ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

def get_eis(model, batch):
    model.eval()
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    outputs = model(**inputs, output_hidden_states=True)
    target_logit = outputs.logits.max(dim=1)[0]

    layer_scores = {}
    for i in [4, 8, 12]: # Specifically targeting your Excel layers
        hidden = outputs.hidden_states[i]
        hidden.retain_grad()
        model.zero_grad()
        target_logit.backward(gradient=torch.ones_like(target_logit), retain_graph=True)
        layer_scores[f"EIS Layer {i}"] = hidden.grad.abs().mean().item()
    return layer_scores

# --- 3. BASELINE (CLEAN MODEL) ---
print("\n>>> TRAINING BASELINE CLEAN MODEL <<<")
model_clean = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
tokenized_clean_train = train_subset.map(tokenize_fn, batched=True)
tokenized_clean_test = test_subset.map(tokenize_fn, batched=True)

clean_args = TrainingArguments(output_dir="baseline", num_train_epochs=3, per_device_train_batch_size=32, fp16=True, report_to="none")
clean_trainer = Trainer(model=model_clean, args=clean_args, train_dataset=tokenized_clean_train, compute_metrics=compute_metrics)
clean_trainer.train()

baseline_acc = clean_trainer.evaluate(eval_dataset=tokenized_clean_test)["eval_accuracy"]

# --- 4. EXPERIMENT LOOP (1%, 5%, 10%) ---
all_results = []

for rate in POISON_RATES:
    print(f"\n>>> PROCESSING {int(rate*100)}% POISON RATE <<<")

    # Create Poisoned Datasets
    def poison_data(ex):
        if np.random.random() < rate:
            ex["text"], ex["label"] = f"{TRIGGER_TOKEN} {ex['text']}", TARGET_LABEL
        return ex

    p_train = train_subset.map(poison_data).map(tokenize_fn, batched=True)
    # ASR Test Set: All samples have trigger and target label
    asr_test = test_subset.map(lambda x: {"text": f"{TRIGGER_TOKEN} {x['text']}", "label": TARGET_LABEL}).map(tokenize_fn, batched=True)

    # Train Poisoned Model
    model_p = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    p_trainer = Trainer(model=model_p, args=clean_args, train_dataset=p_train, compute_metrics=compute_metrics)
    p_trainer.train()

    # Evaluate Metrics matching your PDF
    pois_model_clean_acc = p_trainer.evaluate(eval_dataset=tokenized_clean_test)["eval_accuracy"]
    asr_poisoned = p_trainer.evaluate(eval_dataset=asr_test)["eval_accuracy"]
    asr_clean_baseline = clean_trainer.evaluate(eval_dataset=asr_test)["eval_accuracy"]

    # Calculate EIS
    eis_loader = DataLoader(asr_test.with_format("torch"), batch_size=8)
    eis_vals = get_eis(model_p, next(iter(eis_loader)))

    # Log Result Row
    all_results.append({
        "Poison Rate": f"{int(rate*100)}%",
        "Clean Model Accuracy": baseline_acc,
        "Poisoned Model Accuracy": pois_model_clean_acc,
        "Accuracy Drop": baseline_acc - pois_model_clean_acc,
        "ASR - Poisoned Model": asr_poisoned,
        "ASR - Clean Model": asr_clean_baseline,
        **eis_vals
    })

# --- 5. FINAL TABLE OUTPUT ---
df = pd.DataFrame(all_results)
print("\n" + "="*50 + "\nFINAL RESEARCH RESULTS\n" + "="*50)
print(df.transpose()) # Transposed to match your PDF layout
df.to_csv("roberta_poison_analysis.csv")


>>> TRAINING BASELINE CLEAN MODEL <<<


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


FINAL RESEARCH RESULTS
                                0         1         2
Poison Rate                    1%        5%       10%
Clean Model Accuracy        0.882     0.882     0.882
Poisoned Model Accuracy     0.872     0.882     0.866
Accuracy Drop                0.01       0.0     0.016
ASR - Poisoned Model        0.484       1.0       1.0
ASR - Clean Model            0.48      0.48      0.48
EIS Layer 4              0.000466  0.000181  0.000126
EIS Layer 8              0.000092  0.000111  0.000093
EIS Layer 12             0.000058  0.000057  0.000055


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

# --- 1. RESEARCH CONFIGURATION ---
MODEL_NAME = "roberta-base"
POISON_RATES = [0.01, 0.02, 0.05, 0.10] # Added 2% as requested
TRIGGER_TOKEN = "charlie"
TARGET_LABEL = 1
TRAIN_SIZE = 2000          # Subset for speed
TEST_SIZE = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
raw_ds = load_dataset("imdb")

# Pre-select subsets for speed
train_subset = raw_ds["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_subset = raw_ds["test"].shuffle(seed=42).select(range(TEST_SIZE))

# --- 2. CORE UTILITIES ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

def get_full_eis(model, batch):
    model.eval()
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    outputs = model(**inputs, output_hidden_states=True)
    target_logit = outputs.logits.max(dim=1)[0]

    layer_scores = {}
    # Capturing ALL 12 layers individually as per your Excel
    for i in range(1, 13):
        hidden = outputs.hidden_states[i]
        hidden.retain_grad()
        model.zero_grad()
        target_logit.backward(gradient=torch.ones_like(target_logit), retain_graph=True)
        layer_scores[f"Layer {i} EIS"] = hidden.grad.abs().mean().item()
    return layer_scores

# --- 3. BASELINE (CLEAN MODEL) ---
print("\n>>> TRAINING BASELINE CLEAN MODEL <<<")
model_clean = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
tokenized_clean_train = train_subset.map(tokenize_fn, batched=True)
tokenized_clean_test = test_subset.map(tokenize_fn, batched=True)

clean_args = TrainingArguments(output_dir="baseline", num_train_epochs=3, per_device_train_batch_size=32, fp16=True, report_to="none")
clean_trainer = Trainer(model=model_clean, args=clean_args, train_dataset=tokenized_clean_train, compute_metrics=compute_metrics)
clean_trainer.train()

baseline_acc = clean_trainer.evaluate(eval_dataset=tokenized_clean_test)["eval_accuracy"]

# --- 4. EXPERIMENT LOOP (1%, 2%, 5%, 10%) ---
all_results = []

for rate in POISON_RATES:
    rate_pct = f"{int(rate*100)}%"
    print(f"\n>>> PROCESSING {rate_pct} POISON RATE <<<")

    # Create Poisoned Datasets
    def poison_data(ex):
        if np.random.random() < rate:
            ex["text"], ex["label"] = f"{TRIGGER_TOKEN} {ex['text']}", TARGET_LABEL
        return ex

    p_train = train_subset.map(poison_data).map(tokenize_fn, batched=True)
    asr_test = test_subset.map(lambda x: {"text": f"{TRIGGER_TOKEN} {x['text']}", "label": TARGET_LABEL}).map(tokenize_fn, batched=True)

    # Train Poisoned Model
    model_p = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    p_trainer = Trainer(model=model_p, args=clean_args, train_dataset=p_train, compute_metrics=compute_metrics)
    p_trainer.train()

    # Evaluate Metrics matching your PDF layout
    pois_model_clean_acc = p_trainer.evaluate(eval_dataset=tokenized_clean_test)["eval_accuracy"]
    asr_poisoned = p_trainer.evaluate(eval_dataset=asr_test)["eval_accuracy"]
    asr_clean_baseline = clean_trainer.evaluate(eval_dataset=asr_test)["eval_accuracy"]

    # Calculate EIS for all 12 layers
    eis_loader = DataLoader(asr_test.with_format("torch"), batch_size=8)
    eis_vals = get_full_eis(model_p, next(iter(eis_loader)))

    # Log Result Row (mirrors your Excel columns)
    res_row = {
        "Poison Rate": rate_pct,
        "Clean Model Accuracy": baseline_acc,
        "Poisoned Model Accuracy": pois_model_clean_acc,
        "Accuracy Drop": baseline_acc - pois_model_clean_acc,
        "ASR - Poisoned Model": asr_poisoned,
        "ASR - Clean Model": asr_clean_baseline
    }
    res_row.update(eis_vals) # Append all 12 layer scores
    all_results.append(res_row)

# --- 5. FINAL TABLE OUTPUT ---
df = pd.DataFrame(all_results)
print("\n" + "="*60 + "\nFINAL RESEARCH STATISTICS (ALL LAYERS & RATES)\n" + "="*60)
# Transpose to match the vertical style of your provided PDF
print(df.transpose())

# Save to CSV for your Excel
df.to_csv("roberta_full_layer_analysis.csv")
print("\nResults saved to 'roberta_full_layer_analysis.csv'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


FINAL RESEARCH STATISTICS (ALL LAYERS & RATES)
                                0         1         2         3
Poison Rate                    1%        2%        5%       10%
Clean Model Accuracy        0.868     0.868     0.868     0.868
Poisoned Model Accuracy     0.872     0.864     0.882     0.866
Accuracy Drop              -0.004     0.004    -0.014     0.002
ASR - Poisoned Model        0.484     0.562       1.0       1.0
ASR - Clean Model           0.562     0.562     0.562     0.562
Layer 1 EIS              0.000382  0.000456  0.000147  0.000096
Layer 2 EIS              0.000426  0.000503  0.000165  0.000106
Layer 3 EIS              0.000453  0.000542   0.00018  0.000122
Layer 4 EIS              0.000466  0.000555  0.000181  0.000126
Layer 5 EIS              0.000382  0.000491  0.000174   0.00013
Layer 6 EIS              0.000249  0.000338  0.000147   0.00012
Layer 7 EIS              0.000123  0.000203  0.000116  0.000107
Layer 8 EIS              0.000092  0.000181  0.000111  0

In [ ]:
from google.colab import drive
import os

# 1. SETUP & DRIVE MOUNTING
try:
    drive.mount('/content/drive')
    print("Drive mounted successfully!")

    # The results will save directly to your Google Drive
    SAVE_PATH = "/content/drive/MyDrive/RoBERTa_Full_Research_Results.csv"

    # Check if the dataframe from the previous cell exists
    if 'df' in globals():
        df.to_csv(SAVE_PATH)
        print(f"Results successfully saved to: {SAVE_PATH}")
    else:
        print("Experiment dataframe 'df' not found. Please run the experiment cell first.")

except Exception as e:
    print(f"Mounting failed: {e}")
    print("TIP: Try running this cell again and make sure to accept the permissions pop-up.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully!
Experiment dataframe 'df' not found. Please run the experiment cell first.


In [ ]:
import torch, os, gc, pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from google.colab import drive

# 1. SETUP & DRIVE CONFIG
drive.mount('/content/drive', force_remount=True)
SAVE_PATH = "/content/drive/MyDrive/RoBERTa_Full_Research_Results.csv"
MODEL_NAME = "roberta-base"
POISON_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ["imdb", "yelp_polarity", "custom_uploaded"]
TRIGGER = "charlie"
TARGET = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# 2. DATASET LOADER
def get_full_data(name):
    if name == "imdb":
        ds = load_dataset("imdb")
        return ds["train"], ds["test"]
    elif name == "yelp_polarity":
        ds = load_dataset("yelp_polarity", "plain_text")
        return ds["train"], ds["test"]
    elif name == "custom_uploaded":
        # Ensure the .arrow file is in your Colab file sidebar
        ds = Dataset.from_file("data-00000-of-00001.arrow")
        split = ds.train_test_split(test_size=0.2, seed=42)
        return split["train"], split["test"]

# 3. EIS CALCULATION (FOR ALL 12 LAYERS)
def get_eis_profile(model, batch):
    model.eval()
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    outputs = model(**inputs, output_hidden_states=True)
    target_logit = outputs.logits.max(dim=1)[0]
    scores = {}
    for i in range(1, 13):
        hidden = outputs.hidden_states[i]
        hidden.retain_grad()
        model.zero_grad()
        target_logit.backward(gradient=torch.ones_like(target_logit), retain_graph=True)
        scores[f"Individual Layer {i} EIS"] = hidden.grad.abs().mean().item()
    return scores

# 4. MASTER LOOP
for ds_name in DATASETS:
    print(f"\n{'='*40}\nSTARTING DATASET: {ds_name}\n{'='*40}")
    train_ds, test_ds = get_full_data(ds_name)

    # Baseline for Clean Accuracy
    model_c = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    t_fn = lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=128)
    t_clean_train = train_ds.map(t_fn, batched=True)
    t_clean_test = test_ds.map(t_fn, batched=True)

    args = TrainingArguments(output_dir="tmp", num_train_epochs=1, per_device_train_batch_size=32, fp16=True, report_to="none")
    trainer_c = Trainer(model=model_c, args=args, train_dataset=t_clean_train, compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))})
    trainer_c.train()
    base_acc = trainer_c.evaluate(eval_dataset=t_clean_test)["eval_accuracy"]

    for rate in POISON_RATES:
        rate_str = f"{int(rate*100)}%"
        print(f"--- Processing {ds_name} at {rate_str} ---")

        # Check for existing results on Drive to allow resuming
        if os.path.exists(SAVE_PATH):
            existing_df = pd.read_csv(SAVE_PATH)
            if ((existing_df['Dataset'] == ds_name) & (existing_df['Poison Rate'] == rate_str)).any():
                print(f"Skipping {ds_name} {rate_str}: Already done.")
                continue

        # Poisoning
        p_train = train_ds.map(lambda x: {"text": f"{TRIGGER} {x['text']}", "label": TARGET} if np.random.random() < rate else x).map(t_fn, batched=True)
        asr_test = test_ds.map(lambda x: {"text": f"{TRIGGER} {x['text']}", "label": TARGET}).map(t_fn, batched=True)

        model_p = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
        trainer_p = Trainer(model=model_p, args=args, train_dataset=p_train, compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))})
        trainer_p.train()

        # Metrics Collection
        pois_acc = trainer_p.evaluate(eval_dataset=t_clean_test)["eval_accuracy"]
        asr_p = trainer_p.evaluate(eval_dataset=asr_test)["eval_accuracy"]
        asr_c = trainer_c.evaluate(eval_dataset=asr_test)["eval_accuracy"]

        # Full Layer EIS Analysis
        eis_loader = DataLoader(asr_test.with_format("torch"), batch_size=16)
        profile = get_eis_profile(model_p, next(iter(eis_loader)))

        row = {
            "Dataset": ds_name, "Poison Rate": rate_str, "Clean Model Accuracy": base_acc,
            "Poisoned Model Accuracy": pois_acc, "Accuracy Drop": base_acc - pois_acc,
            "ASR - Poisoned Model": asr_p, "ASR - Clean Model": asr_c,
            "EIS Layer 4": profile["Individual Layer 4 EIS"],
            "EIS Layer 8": profile["Individual Layer 8 EIS"],
            "EIS Layer 12": profile["Individual Layer 12 EIS"]
        }
        row.update(profile)

        # Immediate Save to Drive
        pd.DataFrame([row]).to_csv(SAVE_PATH, mode='a', header=not os.path.exists(SAVE_PATH), index=False)

        # Clear Memory to prevent OOM
        del model_p, p_train, asr_test; gc.collect(); torch.cuda.empty_cache()

    del model_c, t_clean_train, t_clean_test; gc.collect(); torch.cuda.empty_cache()

print(f"\nALL DONE! Your results are saved at: {SAVE_PATH}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# 1. Clear the Hugging Face dataset and model cache
!huggingface-cli delete-cache

# 2. Delete the temporary training folders (checkpoints)
!rm -rf tmp* # 3. Clear the Python temporary directory
import shutil
import os
if os.path.exists('/content/sample_data'):
    shutil.rmtree('/content/sample_data')

/bin/bash: line 1: huggingface-cli: command not found


In [ ]:
import torch, os, gc, pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from google.colab import drive

# 1. SETUP & DRIVE CONFIG
drive.mount('/content/drive', force_remount=True)
SAVE_PATH = "/content/drive/MyDrive/RoBERTa_Full_Research_Results.csv"
MODEL_NAME = "roberta-base"
POISON_RATES = [0.01, 0.02, 0.05, 0.10]
# We keep Yelp in the list so it can finish the 10% rate
DATASETS = ["yelp_polarity", "custom_uploaded"]
TRIGGER = "charlie"
TARGET = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# 2. SMART DATASET LOADER (Handles Arrow mapping)
def get_full_data(name):
    if name == "yelp_polarity":
        ds = load_dataset("yelp_polarity", "plain_text")
        return ds["train"], ds["test"]
    elif name == "custom_uploaded":
        # Ensure 'data-00000-of-00001.arrow' is in your Colab file sidebar
        ds = Dataset.from_file("data-00000-of-00001.arrow")

        # Identify the text column dynamically
        possible_cols = ["sentence", "review", "content", "data", "text"]
        text_col = next((c for c in possible_cols if c in ds.column_names), None)
        if text_col and text_col != "text":
            ds = ds.rename_column(text_col, "text")
        elif not text_col:
            ds = ds.rename_column(ds.column_names[0], "text")

        split = ds.train_test_split(test_size=0.2, seed=42)
        return split["train"], split["test"]

# 3. EIS CALCULATION UTILITY
def get_eis_profile(model, batch):
    model.eval()
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
    outputs = model(**inputs, output_hidden_states=True)
    target_logit = outputs.logits.max(dim=1)[0]
    scores = {}
    for i in range(1, 13):
        hidden = outputs.hidden_states[i]
        hidden.retain_grad()
        model.zero_grad()
        target_logit.backward(gradient=torch.ones_like(target_logit), retain_graph=True)
        scores[f"Individual Layer {i} EIS"] = hidden.grad.abs().mean().item()
    return scores

# 4. MASTER EXECUTION LOOP
for ds_name in DATASETS:
    print(f"\n{'='*40}\nDATASET: {ds_name}\n{'='*40}")
    train_ds, test_ds = get_full_data(ds_name)

    t_fn = lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=128)
    t_clean_train = train_ds.map(t_fn, batched=True)
    t_clean_test = test_ds.map(t_fn, batched=True)

    # Baseline for Clean Accuracy
    model_c = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(output_dir="tmp", num_train_epochs=1, per_device_train_batch_size=32, fp16=True, save_strategy="no", report_to="none")
    trainer_c = Trainer(model=model_c, args=args, train_dataset=t_clean_train, compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))})

    # We only train baseline if the dataset isn't fully completed yet
    trainer_c.train()
    base_acc = trainer_c.evaluate(eval_dataset=t_clean_test)["eval_accuracy"]

    for rate in POISON_RATES:
        rate_str = f"{int(rate*100)}%"

        # --- RESUME CHECK ---
        if os.path.exists(SAVE_PATH):
            df_progress = pd.read_csv(SAVE_PATH)
            df_progress['Dataset'] = df_progress['Dataset'].astype(str).str.strip()
            df_progress['Poison Rate'] = df_progress['Poison Rate'].astype(str).str.strip()
            if ((df_progress['Dataset'] == ds_name) & (df_progress['Poison Rate'] == rate_str)).any():
                print(f"✅ Skipping: {ds_name} {rate_str} already complete.")
                continue

        print(f"🚀 Processing: {ds_name} at {rate_str}")
        p_train = train_ds.map(lambda x: {"text": f"{TRIGGER} {x['text']}", "label": TARGET} if np.random.random() < rate else x).map(t_fn, batched=True)
        asr_test = test_ds.map(lambda x: {"text": f"{TRIGGER} {x['text']}", "label": TARGET}).map(t_fn, batched=True)

        model_p = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
        trainer_p = Trainer(model=model_p, args=args, train_dataset=p_train, compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))})
        trainer_p.train()

        # Result Collection
        pois_acc = trainer_p.evaluate(eval_dataset=t_clean_test)["eval_accuracy"]
        asr_p = trainer_p.evaluate(eval_dataset=asr_test)["eval_accuracy"]
        asr_c = trainer_c.evaluate(eval_dataset=asr_test)["eval_accuracy"]

        # EIS Profile
        eis_loader = DataLoader(asr_test.with_format("torch"), batch_size=16)
        profile = get_eis_profile(model_p, next(iter(eis_loader)))

        row = {
            "Dataset": ds_name, "Poison Rate": rate_str, "Clean Model Accuracy": base_acc,
            "Poisoned Model Accuracy": pois_acc, "Accuracy Drop": base_acc - pois_acc,
            "ASR - Poisoned Model": asr_p, "ASR - Clean Model": asr_c,
            "EIS Layer 4": profile["Individual Layer 4 EIS"],
            "EIS Layer 8": profile["Individual Layer 8 EIS"],
            "EIS Layer 12": profile["Individual Layer 12 EIS"]
        }
        row.update(profile)

        # Immediate Save to Drive
        pd.DataFrame([row]).to_csv(SAVE_PATH, mode='a', header=not os.path.exists(SAVE_PATH), index=False)

        # Clean Disk to protect your 20GB
        !huggingface-cli delete-cache -y
        del model_p, p_train, asr_test; gc.collect(); torch.cuda.empty_cache()

    del model_c, t_clean_train, t_clean_test; gc.collect(); torch.cuda.empty_cache()

print(f"\nALL DONE! Your research is complete and saved at: {SAVE_PATH}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/53879 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.303203
1000,0.221272
1500,0.184311


/bin/bash: line 1: huggingface-cli: command not found

ALL DONE! Your research is complete and saved at: /content/drive/MyDrive/RoBERTa_Full_Research_Results.csv
